# 84 - Avaliação dos experimentos selecionados: NCM e revisão manual

**Pergunta:** os clusters das nove configurações da Tabela 4 do TCC II reúnem produtos do mesmo tipo? A análise usa duas evidências complementares:

1. **Referência externa por NCM.** Cada descrição tem o NCM mais frequente (`codigo_ncm_mais_frequente`) na base `2025_NCMs_filtrados_semstopwords`. O NCM não foi usado no agrupamento, então serve de rótulo fraco e independente. Ele não é a categoria final: NCMs diferentes podem formar a mesma categoria comercial, e a mesma descrição pode aparecer com NCMs distintos nas notas.
2. **Amostra estratificada para revisão manual.** Clusters sorteados por estrato de tamanho, com descrições apresentadas sem identificar o experimento, para avaliação com critério fixo no Excel.

**Origem:** CSVs de `reports/tcc2_selecionados/` (gerados pelo notebook 83) e Parquet `data/processed/2025_NCMs_filtrados_semstopwords/2025_NCMs_filtrados_semstopwords-202501-202512.parquet`, unidos por `descricao_normalizada`.

**Métricas por experimento (somente descrições agrupadas, sem ruído):**

- `pureza`: fração das descrições agrupadas que pertence ao NCM majoritário do seu cluster. Cresce com o número de clusters, por isso é acompanhada de `pureza_aleatoria`, a pureza obtida embaralhando os NCMs entre as mesmas descrições, com os mesmos tamanhos de cluster (semente fixa).
- `ganho_pureza`: `pureza - pureza_aleatoria`, a parte da pureza que não se explica pelo tamanho e pela quantidade de clusters.
- `nmi`, `ami`, `ari`, `homogeneidade`, `completude` entre cluster e NCM (scikit-learn). `ami` e `ari` são ajustadas ao acaso; `nmi` e `homogeneidade` tendem a subir com mais clusters.
- Distribuição da pureza por cluster e cobertura ponderada pela frequência (volume de ocorrências agrupado).

Como as métricas ignoram o ruído, uma configuração que agrupa pouco pode parecer pura por agrupar apenas casos fáceis. Elas devem ser lidas junto da cobertura.

**Saídas** em `reports/tcc2_avaliacao/`: `metricas_ncm.xlsx` (regravável) e o par `revisao_manual.xlsx` / `revisao_chave.csv`, que **nunca é sobrescrito**, para não perder avaliações preenchidas.

In [1]:
from pathlib import Path
import logging

import numpy as np
import pandas as pd
from IPython.display import display
from openpyxl.formatting.rule import FormulaRule
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.worksheet.table import Table, TableStyleInfo
from sklearn.metrics import (
    adjusted_mutual_info_score,
    adjusted_rand_score,
    homogeneity_completeness_v_measure,
    normalized_mutual_info_score,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("notebook84")
pd.set_option("display.max_colwidth", 120)

## Configurações

- `EXPERIMENTS`: rótulo curto e arquivo de cada configuração da Tabela 4, na mesma numeração do notebook 83.
- Amostra: em cada experimento, os clusters são divididos em tercis de tamanho (pequeno, médio, grande) e `CLUSTERS_PER_STRATUM` são sorteados por estrato. Experimentos com até `3 * CLUSTERS_PER_STRATUM` clusters entram por inteiro.
- Cada cluster sorteado mostra as `TOP_N_DESCRIPTIONS` descrições mais frequentes e mais `RANDOM_N_DESCRIPTIONS` sorteadas entre as demais, para não avaliar só os itens mais comuns.
- `BLIND_REVIEW = True`: os clusters recebem códigos (`R001`, ...) em ordem embaralhada, e a planilha de revisão não mostra o experimento nem o NCM. A correspondência fica em `revisao_chave.csv`, que só deve ser aberto depois da avaliação.

In [2]:
PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "data" / "processed").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Não encontrei a raiz do repositório com data/processed/.")

ASSIGNMENTS_DIR = "reports/tcc2_selecionados"
NCM_DATASET_PATH = "data/processed/2025_NCMs_filtrados_semstopwords/2025_NCMs_filtrados_semstopwords-202501-202512.parquet"
OUTPUT_DIR = "reports/tcc2_avaliacao"
METRICS_EXCEL = "metricas_ncm.xlsx"
REVIEW_EXCEL = "revisao_manual.xlsx"
REVIEW_KEY = "revisao_chave.csv"

EXPERIMENTS = {
    "01 TFIDF KMeans": "01_tfidf_kmeans_word_2_5_min_df10_k200.csv",
    "02 MPNet KMeans": "02_mpnet_kmeans_k200.csv",
    "03 TFIDF DBSCAN": "03_tfidf_dbscan_word_2_5_min_df10_eps0.025_min_samples3.csv",
    "04 MPNet DBSCAN": "04_mpnet_dbscan_eps0.01_min_samples5.csv",
    "05 TFIDF HDBSCAN 1o": "05_tfidf_hdbscan_1o_char_wb_5_10_min_df10_mcs5_min_samplesdefault.csv",
    "06 TFIDF HDBSCAN A": "06_tfidf_hdbscan_candidata_A_word_2_5_min_df10_mcs20_min_samplesdefault.csv",
    "07 TFIDF HDBSCAN B": "07_tfidf_hdbscan_candidata_B_char_wb_5_10_min_df10_mcs5_min_samples10.csv",
    "08 MPNet HDBSCAN": "08_mpnet_hdbscan_mcs5_min_samples10.csv",
    "09 Hibrido HDBSCAN": "09_tfidf_mpnet_hdbscan_alpha0.2_word_2_5_min_df5_mcs5_min_samples2.csv",
}
EXPECTED_ROWS = 20_000
NOISE_LABEL = -1

RANDOM_SEED = 42
CLUSTERS_PER_STRATUM = 7
TOP_N_DESCRIPTIONS = 5
RANDOM_N_DESCRIPTIONS = 5
BLIND_REVIEW = True
PURE_THRESHOLD = 0.8
MIXED_THRESHOLD = 0.4

REVIEW_OPTIONS = ["coerente", "parcialmente coerente", "incoerente"]
REVIEW_CRITERIA = [
    "coerente: todas as descrições mostradas são do mesmo tipo de produto (ex.: todas são açúcar).",
    "parcialmente coerente: a maioria é do mesmo tipo, com uma ou duas descrições de outro tipo, "
    "ou o grupo é amplo demais (ex.: condimentos variados).",
    "incoerente: as descrições não formam um tipo de produto reconhecível ou se agrupam por "
    "termos genéricos (ex.: 'aspecto fisico', 'tipo').",
    "Avalie apenas pelo texto das descrições; não abra revisao_chave.csv antes de terminar.",
]
GROUP_COLORS = ("DDEBF7", "B4C6E7")

## Funções

In [3]:
def load_experiments(ncm_table):
    """Lê os assignments e anexa o NCM mais frequente de cada descrição."""
    loaded = {}
    for label, file_name in EXPERIMENTS.items():
        assignments = pd.read_csv(PROJECT_ROOT / ASSIGNMENTS_DIR / file_name)
        merged = assignments.merge(
            ncm_table, on="descricao_normalizada", how="left", validate="one_to_one", suffixes=("", "_base")
        )
        missing_ncm = int(merged["ncm"].isna().sum())
        if len(merged) != EXPECTED_ROWS or missing_ncm:
            raise ValueError(f"{label}: {len(merged)} linhas, {missing_ncm} sem NCM")
        if not merged["frequencia"].equals(merged["frequencia_base"]):
            raise ValueError(f"{label}: frequência diverge da base processada")
        loaded[label] = merged.drop(columns="frequencia_base")
        logger.info("%s: %d descrições carregadas", label, len(merged))
    return loaded


def cluster_purity_table(grouped):
    """Pureza NCM, NCM majoritário e tamanho de cada cluster."""
    counts = grouped.groupby(["cluster", "ncm"]).size().rename("n").reset_index()
    sizes = counts.groupby("cluster")["n"].sum().rename("tamanho")
    majority = counts.sort_values(["cluster", "n"], ascending=[True, False]).drop_duplicates("cluster")
    table = majority.set_index("cluster").join(sizes)
    table["ncms_distintos"] = counts.groupby("cluster").size()
    table["pureza"] = table["n"] / table["tamanho"]
    return table.rename(columns={"ncm": "ncm_majoritario", "n": "n_ncm_majoritario"}).reset_index()


def weighted_purity(labels, ncm):
    """Pureza global: fração dos itens que pertence ao NCM majoritário do seu cluster."""
    counts = pd.crosstab(labels, ncm)
    return counts.max(axis=1).sum() / counts.values.sum()


def experiment_metrics(label, frame, rng):
    """Calcula as métricas NCM de um experimento sobre as descrições agrupadas."""
    grouped = frame.loc[frame["cluster"].ne(NOISE_LABEL)]
    labels, ncm = grouped["cluster"].to_numpy(), grouped["ncm"].to_numpy()
    purity_by_cluster = cluster_purity_table(grouped)
    purity = weighted_purity(labels, ncm)
    random_purity = weighted_purity(labels, rng.permutation(ncm))
    homogeneity, completeness, _ = homogeneity_completeness_v_measure(ncm, labels)
    logger.info("%s: métricas calculadas", label)
    return {
        "experimento": label,
        "clusters": int(grouped["cluster"].nunique()),
        "agrupadas": len(grouped),
        "cobertura_pct": round(len(grouped) / len(frame) * 100, 3),
        "cobertura_frequencia_pct": round(grouped["frequencia"].sum() / frame["frequencia"].sum() * 100, 3),
        "ncms_agrupados": int(grouped["ncm"].nunique()),
        "pureza": purity,
        "pureza_aleatoria": random_purity,
        "ganho_pureza": purity - random_purity,
        "pureza_mediana_cluster": purity_by_cluster["pureza"].median(),
        f"clusters_pureza_ge_{PURE_THRESHOLD}": int(purity_by_cluster["pureza"].ge(PURE_THRESHOLD).sum()),
        f"clusters_pureza_lt_{MIXED_THRESHOLD}": int(purity_by_cluster["pureza"].lt(MIXED_THRESHOLD).sum()),
        "nmi": normalized_mutual_info_score(ncm, labels),
        "ami": adjusted_mutual_info_score(ncm, labels),
        "ari": adjusted_rand_score(ncm, labels),
        "homogeneidade": homogeneity,
        "completude": completeness,
    }, purity_by_cluster.assign(experimento=label)


def sample_clusters(label, frame, rng):
    """Sorteia clusters por tercil de tamanho e escolhe as descrições a revisar."""
    grouped = frame.loc[frame["cluster"].ne(NOISE_LABEL)]
    sizes = grouped.groupby("cluster").size()
    strata = pd.qcut(sizes.rank(method="first"), 3, labels=["pequeno", "medio", "grande"])
    picked = []
    for stratum, members in sizes.groupby(strata, observed=True):
        clusters = members.index.to_numpy()
        take = min(CLUSTERS_PER_STRATUM, len(clusters))
        picked += [(cluster, stratum) for cluster in rng.choice(clusters, size=take, replace=False)]
    rows = []
    for cluster, stratum in picked:
        members = grouped.loc[grouped["cluster"].eq(cluster)].sort_values("frequencia", ascending=False)
        top = members.head(TOP_N_DESCRIPTIONS)
        rest = members.iloc[TOP_N_DESCRIPTIONS:]
        extra = rest.sample(n=min(RANDOM_N_DESCRIPTIONS, len(rest)), random_state=int(rng.integers(1 << 31)))
        shown = pd.concat([top.assign(selecao="mais frequente"), extra.assign(selecao="sorteada")])
        rows.append(shown.assign(experimento=label, estrato=stratum, tamanho=len(members)))
    return pd.concat(rows, ignore_index=True)


def format_sheet(worksheet, frame, table_name, wrap_columns=()):
    """Formata a aba como Tabela do Excel com cabeçalho fixo e larguras ajustadas."""
    last_cell = f"{get_column_letter(frame.shape[1])}{len(frame) + 1}"
    table = Table(displayName=table_name, ref=f"A1:{last_cell}")
    table.tableStyleInfo = TableStyleInfo(name="TableStyleMedium2", showRowStripes=not wrap_columns)
    worksheet.add_table(table)
    worksheet.freeze_panes = "A2"
    for index, column in enumerate(frame.columns, start=1):
        width = max(len(str(column)), frame[column].astype(str).str.len().quantile(0.95))
        worksheet.column_dimensions[get_column_letter(index)].width = min(max(width + 2, 10), 90)
        if column in wrap_columns:
            for cell in worksheet[get_column_letter(index)][1:]:
                cell.alignment = Alignment(wrap_text=True, vertical="top")


def color_by_group(worksheet, frame, group_column):
    """Alterna dois tons entre grupos vizinhos usando a paridade de uma coluna numérica."""
    column = get_column_letter(frame.columns.get_loc(group_column) + 1)
    cells = f"A2:{get_column_letter(frame.shape[1])}{len(frame) + 1}"
    for formula, color in [(f"ISODD(${column}2)", GROUP_COLORS[1]), (f"ISEVEN(${column}2)", GROUP_COLORS[0])]:
        fill = PatternFill(start_color=color, end_color=color, fill_type="solid")
        worksheet.conditional_formatting.add(cells, FormulaRule(formula=[formula], fill=fill, stopIfTrue=True))

## Execução

### 1. Carregar assignments e NCM

In [4]:
ncm_table = pd.read_parquet(
    PROJECT_ROOT / NCM_DATASET_PATH,
    columns=["descricao_normalizada", "frequencia", "descricao_original_exemplo",
             "codigo_ncm_mais_frequente", "ncm_tipo_mais_frequente"],
).rename(columns={"frequencia": "frequencia_base", "ncm_tipo_mais_frequente": "ncm_descricao"})
ncm_table["ncm"] = ncm_table.pop("codigo_ncm_mais_frequente").astype("Int64").astype(str).str.zfill(8)
loaded = load_experiments(ncm_table)

2026-10-26 11:22:40,417 INFO 01 TFIDF KMeans: 20000 descrições carregadas


2026-10-26 11:22:40,530 INFO 02 MPNet KMeans: 20000 descrições carregadas


2026-10-26 11:22:40,648 INFO 03 TFIDF DBSCAN: 20000 descrições carregadas


2026-10-26 11:22:40,770 INFO 04 MPNet DBSCAN: 20000 descrições carregadas


2026-10-26 11:22:40,881 INFO 05 TFIDF HDBSCAN 1o: 20000 descrições carregadas


2026-10-26 11:22:40,999 INFO 06 TFIDF HDBSCAN A: 20000 descrições carregadas


2026-10-26 11:22:41,121 INFO 07 TFIDF HDBSCAN B: 20000 descrições carregadas


2026-10-26 11:22:41,232 INFO 08 MPNet HDBSCAN: 20000 descrições carregadas


2026-10-26 11:22:41,347 INFO 09 Hibrido HDBSCAN: 20000 descrições carregadas


### 2. Métricas NCM por experimento

In [5]:
rng = np.random.default_rng(RANDOM_SEED)
metric_rows, cluster_tables = [], []
for label, frame in loaded.items():
    row, clusters = experiment_metrics(label, frame, rng)
    metric_rows.append(row)
    cluster_tables.append(clusters)

metrics = pd.DataFrame(metric_rows)
clusters_ncm = pd.concat(cluster_tables, ignore_index=True)
ncm_names = ncm_table.drop_duplicates("ncm").set_index("ncm")["ncm_descricao"]
clusters_ncm["ncm_majoritario_descricao"] = clusters_ncm["ncm_majoritario"].map(ncm_names)
clusters_ncm = clusters_ncm[[
    "experimento", "cluster", "tamanho", "ncms_distintos", "ncm_majoritario",
    "ncm_majoritario_descricao", "n_ncm_majoritario", "pureza",
]]
display(metrics.round(4))

2026-10-26 11:22:41,441 INFO 01 TFIDF KMeans: métricas calculadas


2026-10-26 11:22:41,676 INFO 02 MPNet KMeans: métricas calculadas


2026-10-26 11:22:41,945 INFO 03 TFIDF DBSCAN: métricas calculadas


2026-10-26 11:22:42,055 INFO 04 MPNet DBSCAN: métricas calculadas


2026-10-26 11:22:42,131 INFO 05 TFIDF HDBSCAN 1o: métricas calculadas


2026-10-26 11:22:42,321 INFO 06 TFIDF HDBSCAN A: métricas calculadas


2026-10-26 11:22:42,409 INFO 07 TFIDF HDBSCAN B: métricas calculadas


2026-10-26 11:22:42,541 INFO 08 MPNet HDBSCAN: métricas calculadas


2026-10-26 11:22:42,681 INFO 09 Hibrido HDBSCAN: métricas calculadas


,experimento,clusters,agrupadas,cobertura_pct,cobertura_frequencia_pct,ncms_agrupados,pureza,pureza_aleatoria,ganho_pureza,pureza_mediana_cluster,clusters_pureza_ge_0.8,clusters_pureza_lt_0.4,nmi,ami,ari,homogeneidade,completude
0,01 TFIDF KMeans,200,20000,100.000,100.000,240,0.2606,0.0730,0.1876,0.7918,99,31,0.3404,0.2405,0.0044,0.2487,0.5391
1,02 MPNet KMeans,200,20000,100.000,100.000,240,0.3732,0.0708,0.3023,0.3814,25,106,0.4846,0.3898,0.1039,0.5082,0.4631
2,03 TFIDF DBSCAN,734,7217,36.085,45.998,218,0.7356,0.1496,0.5860,0.8000,376,122,0.7237,0.5403,0.1873,0.8523,0.6288
3,04 MPNet DBSCAN,11,133,0.665,0.220,9,1.0000,0.4586,0.5414,1.0000,11,0,0.9510,0.9413,0.9477,1.0000,0.9066
4,05 TFIDF HDBSCAN 1o,736,10183,50.915,69.098,229,0.6907,0.1245,0.5661,0.7500,347,112,0.7021,0.5263,0.1511,0.8203,0.6137
5,06 TFIDF HDBSCAN A,101,4001,20.005,27.443,176,0.6708,0.0940,0.5769,0.7500,42,17,0.7501,0.6734,0.3641,0.7557,0.7445
6,07 TFIDF HDBSCAN B,381,7947,39.735,58.761,218,0.6927,0.1063,0.5864,0.7167,158,68,0.7330,0.6062,0.2829,0.8079,0.6709
7,08 MPNet HDBSCAN,123,3381,16.905,24.153,178,0.6809,0.0926,0.5883,0.8077,65,20,0.8068,0.7320,0.5027,0.7922,0.8220
8,09 Hibrido HDBSCAN,982,9671,48.355,62.958,230,0.7482,0.1501,0.5981,0.8170,567,87,0.7129,0.5091,0.1181,0.8594,0.6091


### 3. Sortear clusters para revisão manual

In [6]:
rng = np.random.default_rng(RANDOM_SEED)
sample = pd.concat([sample_clusters(label, frame, rng) for label, frame in loaded.items()], ignore_index=True)

keys = sample[["experimento", "cluster", "estrato", "tamanho"]].drop_duplicates().reset_index(drop=True)
order = rng.permutation(len(keys)) if BLIND_REVIEW else np.arange(len(keys))
keys = keys.iloc[order].reset_index(drop=True)
keys.insert(0, "codigo", [f"R{i:03d}" for i in range(1, len(keys) + 1)])
keys.insert(1, "ordem", range(1, len(keys) + 1))
keys = keys.merge(clusters_ncm[["experimento", "cluster", "ncm_majoritario", "pureza"]], on=["experimento", "cluster"])

review = sample.merge(keys[["codigo", "ordem", "experimento", "cluster"]], on=["experimento", "cluster"])
review = review.sort_values(["ordem", "selecao", "frequencia"], ascending=[True, True, False])
review["posicao"] = review.groupby("codigo").cumcount() + 1
review = review[["ordem", "codigo", "tamanho", "posicao", "selecao", "descricao_normalizada",
                 "descricao_original_exemplo", "frequencia"]]
if not BLIND_REVIEW:
    review = review.merge(keys[["codigo", "experimento", "cluster"]], on="codigo")

evaluation = keys[["ordem", "codigo", "tamanho"]].assign(avaliacao="", observacao="")
display(keys.groupby(["experimento", "estrato"], observed=True).size().unstack())

estrato,grande,medio,pequeno
experimento,,,
01 TFIDF KMeans,7,7,7
02 MPNet KMeans,7,7,7
03 TFIDF DBSCAN,7,7,7
04 MPNet DBSCAN,4,3,4
05 TFIDF HDBSCAN 1o,7,7,7
06 TFIDF HDBSCAN A,7,7,7
07 TFIDF HDBSCAN B,7,7,7
08 MPNet HDBSCAN,7,7,7
09 Hibrido HDBSCAN,7,7,7


### 4. Gravar as saídas

**Grava arquivos** em `reports/tcc2_avaliacao/`. `metricas_ncm.xlsx` é regravado a cada execução. `revisao_manual.xlsx` e `revisao_chave.csv` só são gravados se nenhum dos dois existir, para preservar avaliações já preenchidas; para gerar uma nova amostra, renomeie ou mova os arquivos anteriores.

In [7]:
output_dir = PROJECT_ROOT / OUTPUT_DIR
output_dir.mkdir(parents=True, exist_ok=True)

metrics_path = output_dir / METRICS_EXCEL
with pd.ExcelWriter(metrics_path, engine="openpyxl") as writer:
    metrics.round(4).to_excel(writer, sheet_name="Metricas", index=False)
    format_sheet(writer.sheets["Metricas"], metrics, "t_metricas")
    clusters_ncm.round(4).to_excel(writer, sheet_name="Clusters", index=False)
    format_sheet(writer.sheets["Clusters"], clusters_ncm, "t_clusters")
logger.info("Métricas gravadas em %s", metrics_path)

review_path, key_path = output_dir / REVIEW_EXCEL, output_dir / REVIEW_KEY
if review_path.exists() or key_path.exists():
    logger.warning("Revisão já existe; mantida sem alterações: %s", review_path)
else:
    keys.to_csv(key_path, index=False)
    with pd.ExcelWriter(review_path, engine="openpyxl") as writer:
        instructions = pd.DataFrame({"instrucoes": [
            f"Avalie cada código da aba Avaliacao escolhendo: {', '.join(REVIEW_OPTIONS)}.",
            "As descrições de cada código estão na aba Descricoes "
            f"({TOP_N_DESCRIPTIONS} mais frequentes e até {RANDOM_N_DESCRIPTIONS} sorteadas).",
            *REVIEW_CRITERIA,
        ]})
        instructions.to_excel(writer, sheet_name="Instrucoes", index=False)
        writer.sheets["Instrucoes"].column_dimensions["A"].width = 140
        writer.sheets["Instrucoes"]["A1"].font = Font(bold=True)

        evaluation.to_excel(writer, sheet_name="Avaliacao", index=False)
        sheet = writer.sheets["Avaliacao"]
        format_sheet(sheet, evaluation, "t_avaliacao")
        sheet.column_dimensions["D"].width = 24
        sheet.column_dimensions["E"].width = 60
        validation = DataValidation(type="list", formula1='"' + ",".join(REVIEW_OPTIONS) + '"', allow_blank=True)
        sheet.add_data_validation(validation)
        validation.add(f"D2:D{len(evaluation) + 1}")

        review.to_excel(writer, sheet_name="Descricoes", index=False)
        format_sheet(writer.sheets["Descricoes"], review, "t_descricoes",
                     wrap_columns=("descricao_original_exemplo",))
        color_by_group(writer.sheets["Descricoes"], review, "ordem")
    logger.info("Revisão gravada em %s e chave em %s", review_path, key_path)

2026-10-26 11:22:43,575 INFO Métricas gravadas em C:\Users\samuel.simao\Sources\Nofis-Classifier\reports\tcc2_avaliacao\metricas_ncm.xlsx


2026-10-26 11:22:43,816 INFO Revisão gravada em C:\Users\samuel.simao\Sources\Nofis-Classifier\reports\tcc2_avaliacao\revisao_manual.xlsx e chave em C:\Users\samuel.simao\Sources\Nofis-Classifier\reports\tcc2_avaliacao\revisao_chave.csv


### 5. Conferência

In [8]:
saved_keys = pd.read_csv(key_path)
saved_review = pd.read_excel(review_path, sheet_name="Descricoes")
if set(saved_review["codigo"]) != set(saved_keys["codigo"]):
    raise ValueError("Códigos da revisão e da chave não coincidem.")
print(f"{len(saved_keys)} clusters para revisar, {len(saved_review)} linhas de descrições")
print(sorted(path.name for path in output_dir.iterdir()))

179 clusters para revisar, 1585 linhas de descrições
['metricas_ncm.xlsx', 'revisao_chave.csv', 'revisao_manual.xlsx']


## Interpretação

- Compare `pureza` sempre com `pureza_aleatoria`: com muitos clusters pequenos, mesmo um agrupamento aleatório tem pureza alta. `ganho_pureza`, `ami` e `ari` são os indicadores mais comparáveis entre experimentos com quantidades diferentes de clusters.
- As métricas desconsideram o ruído. Leia-as junto de `cobertura_pct` e `cobertura_frequencia_pct`.
- Pureza NCM baixa não prova incoerência: um cluster de "biscoitos" pode reunir NCMs diferentes e ainda ser uma boa categoria. Pureza alta também não basta: o cluster pode separar um mesmo produto em vários grupos (veja `completude`).
- Depois de preencher a aba `Avaliacao`, una-a a `revisao_chave.csv` por `codigo` para contar, por experimento e por estrato, quantos clusters foram considerados coerentes, e compare com a pureza NCM de cada cluster sorteado.